# Feature Engineering with Pandas UDF

mapInPandas / applyInPandas for distributed feature engineering.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import rand

spark = (SparkSession.builder
    .appName('pandas-udf-feature')
    .config('spark.hadoop.fs.s3a.endpoint', 'http://minio:9000')
    .config('spark.hadoop.fs.s3a.access.key', os.environ.get('AWS_ACCESS_KEY_ID', ''))
    .config('spark.hadoop.fs.s3a.secret.key', os.environ.get('AWS_SECRET_ACCESS_KEY', ''))
    .config('spark.hadoop.fs.s3a.path.style.access', 'true')
    .getOrCreate())

df = spark.range(5000).withColumn('x', rand(42)).withColumn('y', rand(17))
df.show(5)

In [ ]:
def add_features(pdf_iter):
    for pdf in pdf_iter:
        pdf['x_squared'] = pdf['x'] ** 2
        pdf['xy'] = pdf['x'] * pdf['y']
        yield pdf

schema = "id long, x double, y double, x_squared double, xy double"
out = df.mapInPandas(add_features, schema)
out.show(5)

In [ ]:
out.write.mode('overwrite').parquet('s3a://spark-jobs/pandas-udf-features/')